# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Davionnic/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane: Refresh / Content Opportunity Scoring.**

I am a backend / search-intelligence intern. The starter pipeline already ranks pages for refresh review, and this lane matches that job: given limited editor time, which pages get looked at first? Freestyle would mean inventing a new question on the gated warehouse; for Week 1 I stay on the bundled 30k-row CSV and treat "which page to review first" as the provisional lane. If warehouse access opens later, the same question can use a real future-window label — for now the starter proxy (`trend_direction == "down"`) is enough to frame the work.


In [1]:
import os
import pandas as pd

# walk up until we find the repo root (works from work/notebooks/ or Colab)
while not os.path.isdir("data/raw") and os.getcwd() != "/":
    os.chdir("..")

RAW = "data/raw/content_refresh_anonymized.csv"
assert os.path.exists(RAW), f"missing starter CSV at {os.path.abspath(RAW)}"
df = pd.read_csv(RAW)
print("Working dir:", os.getcwd())
print(f"starter rows={len(df):,}  cols={df.shape[1]}  clients={df['client_id'].nunique()}")
print("lane locked: Refresh / Content Opportunity Scoring (starter CSV only — no HF warehouse this week)")


Working dir: /workspace/flyrank-ml-internship
starter rows=30,000  cols=44  clients=32
lane locked: Refresh / Content Opportunity Scoring (starter CSV only — no HF warehouse this week)


## 2. The question: decision, action, cost of a wrong call

**Research question:** Which pages should a content / SEO reviewer look at first for refresh, expansion, protection, pruning, or monitoring?

**Decision:** given a long catalog and a short workday, which page gets the next review slot?

**Who acts:** an editor or SEO specialist. They open the ranked queue, read the reason codes, and either refresh the page, expand thin spots, fix CTR/title issues, prune, or park it as monitor.

**Cost of a wrong call:**
- False positive (rank a healthy page high): wasted editor hours on pages that did not need work.
- False negative (bury a real decline): keep burning impressions on a page that is already sliding, while capacity goes elsewhere.

A plain if-statement baseline exists in the starter (`stale_visible_page`, `declining_with_demand`, etc.). ML earns a seat only if a learned ranker beats that rule on a metric that matches the decision — Precision@K for a top-K review queue — under a client-holdout split.


In [2]:
# Decision framing numbers from the starter slice (proxy label, not a future outcome)
n = len(df)
down = (df["trend_direction"] == "down").sum()
demand = df["impressions_90d"] >= 100
declining_with_demand = ((df["trend_direction"] == "down") & demand).sum()
print(f"pages: {n:,}")
print(f"proxy declining (trend_direction=='down'): {down:,} ({down/n:.1%})")
print(f"declining WITH demand (impressions_90d>=100): {declining_with_demand:,} ({declining_with_demand/n:.1%})")
print("wrong call cost in plain terms: review capacity is tiny vs ~13k demand+decline candidates")


pages: 30,000
proxy declining (trend_direction=='down'): 16,262 (54.2%)
declining WITH demand (impressions_90d>=100): 13,152 (43.8%)
wrong call cost in plain terms: review capacity is tiny vs ~13k demand+decline candidates


## 3. Quick look at the data (2-3 real numbers)

Three numbers from the bundled starter CSV that make this lane worth the next weeks:

1. **54.2%** of pages are tagged `trend_direction == "down"` (16,262 / 30,000) — decline is common, so a random pick is a coin flip.
2. **13,152 pages** are declining *and* have `impressions_90d >= 100` — enough demand that a bad pick wastes real visibility, not noise.
3. **~51%** of all impressions in this slice sit on currently-down pages — most of the traffic stake is in the declining half.

These are observed counts on the anonymized starter. They are not proof that a refresh will fix anything. They are the reason a ranked review queue beats scrolling the catalog by hand.


In [3]:
n = len(df)
down_mask = df["trend_direction"] == "down"
down_n = int(down_mask.sum())
decl_demand_n = int((down_mask & (df["impressions_90d"] >= 100)).sum())
imp_total = df["impressions_90d"].sum()
imp_on_down = df.loc[down_mask, "impressions_90d"].sum()

print("--- ML-02 lane evidence (starter CSV) ---")
print(f"1) declining rate: {down_n:,} / {n:,} = {down_n/n:.1%}")
print(f"2) declining + impressions_90d>=100: {decl_demand_n:,}")
print(f"3) share of impressions on down pages: {imp_on_down/imp_total:.1%} "
      f"({imp_on_down:,.0f} / {imp_total:,.0f})")

# one extra check: page-1 pages with age that still need a human look
page1 = (df["avg_position"] > 0) & (df["avg_position"] <= 10) & (df["impressions_90d"] >= 500)
print(f"bonus) page-1 visible (pos<=10, imp>=500): {int(page1.sum()):,} "
      f"| of those down: {int((page1 & down_mask).sum()):,}")


--- ML-02 lane evidence (starter CSV) ---
1) declining rate: 16,262 / 30,000 = 54.2%
2) declining + impressions_90d>=100: 13,152
3) share of impressions on down pages: 51.3% (79,994,363 / 156,010,989)
bonus) page-1 visible (pos<=10, imp>=500): 7,564 | of those down: 4,454


## 4. Careful words: what I can and can't claim

**I can claim (observed / directional / decision-support):**
- On this starter slice, page X ranks above page Y for *review priority* under a stated score and validation split.
- A model beat (or failed to beat) a transparent baseline on Precision@K / ROC-AUC — measured numbers, not vibes.
- Reason codes explain *why* a page was surfaced (stale + visible, declining with demand, low CTR at a given position tier, etc.).

**I cannot claim:**
- That a refresh *caused* a recovery (needs an experiment or causal design; this CSV alone cannot prove it).
- That I predicted Google's algorithm, rankings factors, or AI citations.
- That the starter proxy label (`trend_direction == "down"` from the same window) is a true future outcome. It is a teaching stand-in. A stronger later label is prior-window features → next-window decline/recovery on the warehouse daily facts — if/when HF access is available.
- Anything that identifies a client: no names, domains, URLs, raw queries, or titles.

Working definition for this provisional lane: build a ranked review queue that helps a human spend limited time better. Decision-support, not automation that "fixes SEO."


In [4]:
# Public-safety sanity check: we only print aggregates / pseudonym counts
private_like = [c for c in df.columns if any(k in c.lower() for k in ("url", "title", "domain", "query", "keyword_text"))]
print("private-like columns present:", private_like if private_like else "none")
print(f"client_id is pseudonym only — distinct count={df['client_id'].nunique()} (never a feature)")
print("label source columns (never features): trend_direction, trend_pct")
print("claims mode for this lane: observed / directional / decision-support")


private-like columns present: none
client_id is pseudonym only — distinct count=32 (never a feature)
label source columns (never features): trend_direction, trend_pct
claims mode for this lane: observed / directional / decision-support


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

**Note:** this week uses only `data/raw/content_refresh_anonymized.csv`. Hugging Face warehouse access is not required for ML-02.
